In [1]:
import cv2
import mediapipe as mp
import numpy as np

print("MediaPipe Version:", mp.__version__)
print("OpenCV Version:", cv2.__version__)

MediaPipe Version: 0.10.35
OpenCV Version: 4.10.0


In [2]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

print("Face Landmarker Loaded!")

Face Landmarker Loaded!


In [3]:
import cv2

cap = cv2.VideoCapture(0)

print(cap.isOpened())

ret, frame = cap.read()

print(ret)

cap.release()

True
True


In [4]:
import cv2
import mediapipe as mp
import numpy as np
import time

cap = cv2.VideoCapture(0)

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    timestamp = int(time.time() * 1000)

    result = landmarker.detect_for_video(mp_image, timestamp)

    if result.face_landmarks:

        landmarks = result.face_landmarks[0]

        h, w = frame.shape[:2]

        nose = landmarks[1]
        left_face = landmarks[234]
        right_face = landmarks[454]
        chin = landmarks[152]
        forehead = landmarks[10]

        nose_x = nose.x * w
        left_x = left_face.x * w
        right_x = right_face.x * w

        nose_y = nose.y * h
        forehead_y = forehead.y * h
        chin_y = chin.y * h

        center_x = (left_x + right_x) / 2
        center_y = (forehead_y + chin_y) / 2

        yaw = nose_x - center_x
        pitch = nose_y - center_y

        if yaw > 25:
            direction = "Looking Right"
        elif yaw < -25:
            direction = "Looking Left"
        elif pitch > 20:
            direction = "Looking Down"
        elif pitch < -20:
            direction = "Looking Up"
        else:
            direction = "Looking Straight"

        cv2.putText(
            frame,
            direction,
            (20,40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,255,0),
            2
        )

        cv2.putText(
            frame,
            f"Yaw: {yaw:.1f}",
            (20,80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255,0,0),
            2
        )

        cv2.putText(
            frame,
            f"Pitch: {pitch:.1f}",
            (20,120),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255,0,0),
            2
        )

    cv2.imshow("Head Pose", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()